# RAG Structured Output
LLM JSON Schema를 원하는 구조로 응답하도록 처리할 수 있다.

`llm.with_structured_output(PydanticModelClass)`

In [1]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')

In [2]:
#가상 검색기
from langchain_core.documents import Document

def retrieve_vectordb(query = None):
    return [
        Document(page_content="파리는 프랑스의 수도로, 연간 관광객 수 약 3000만 명입니다. 주요 관광지로는 센강, 개선문, 루브르 박물관이 있습니다."),  # 파리 문서
        Document(page_content="런던은 영국의 수도로, 연간 관광객 수 약 2000만 명입니다. 주요 관광지로는 버킹엄 궁전, 런던 아이, 타워 브릿지가 있습니다."),  # 런던 문서
        Document(page_content="교토는 일본의 옛 수도로, 연간 관광객 수 약 1500만 명입니다. 주요 관광지로는 금각사, 은각사, 기요미즈데라가 있습니다.")  # 교토 문서
    ]
retrieve_vectordb('파리')

[Document(metadata={}, page_content='파리는 프랑스의 수도로, 연간 관광객 수 약 3000만 명입니다. 주요 관광지로는 센강, 개선문, 루브르 박물관이 있습니다.'),
 Document(metadata={}, page_content='런던은 영국의 수도로, 연간 관광객 수 약 2000만 명입니다. 주요 관광지로는 버킹엄 궁전, 런던 아이, 타워 브릿지가 있습니다.'),
 Document(metadata={}, page_content='교토는 일본의 옛 수도로, 연간 관광객 수 약 1500만 명입니다. 주요 관광지로는 금각사, 은각사, 기요미즈데라가 있습니다.')]

In [8]:
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field # 출력 스키마 검증/구조화
from typing import List 
from langchain.chat_models import init_chat_model

class CityInfo(BaseModel):
    city: str = Field(description='도시 이름')
    visitors: int = Field(description='연간 방문객 수')
    landmarks: List[str] = Field(description='주요 관광지 목록')

class CityList(BaseModel):
    cities: List[CityInfo]

llm = init_chat_model('gpt-5.6-luna')
prompt = PromptTemplate.from_template('''
당신은 조회된 문서에서 사용자 원하는 정보를 추출하는 에이젼트입니다.
제공된 문서를 바탕으로 JSON형식으로 답변해주세요.

[조회된 문서]
{docs}

[사용자 질문]
{query}
''')

chain = prompt | llm.with_structured_output(CityList)

context = '\n\n'.join([doc.page_content for doc in retrieve_vectordb()])
query = '교토의 주요 관광 정보'
response: CityList = chain.invoke({'docs': context, 'query':query})
print(response)



cities=[CityInfo(city='교토', visitors=15000000, landmarks=['금각사', '은각사', '기요미즈데라'])]


### Pydantic 문법

In [13]:
from pydantic import BaseModel, Field
from typing import List, Dict, Annotated  # 타입 힌트
from datetime import datetime


# Pydantic 모델 클래스 : 사용자 데이터를 검증/파싱
class User(BaseModel):
    id: int
    name: str
    email: str = None  # 기본값 None (옵션)
    is_active: bool = True  # 기본값 True
    created_at: datetime


honggd = User(
    id=1,
    name='홍길동',
    email='honggd@naver.com',
    is_active=True,
    created_at=datetime.now()
)

print(honggd)

id=1 name='홍길동' email='honggd@naver.com' is_active=True created_at=datetime.datetime(2026, 9, 2, 16, 19, 26, 317347)


In [14]:
# id는 문자형 -> 숫자형 자동파싱, 기본값(옵션)은 기본설정
sinsa = User(id = "2", name = "신사임당", created_at = datetime.now())
print(sinsa)

id=2 name='신사임당' email=None is_active=True created_at=datetime.datetime(2026, 9, 2, 16, 19, 30, 190292)


In [16]:
class Product(BaseModel):
    code: str = Field(..., description = '제품식별코드')
    name: str = Field(..., description = '제품명', min_length = 1, max_length = 10)
    price: float = Field(..., description = '제품가격', ge = 0)
    quantity: int = Field(default = 0, description = '제품수량')
    description: str = Field(..., description = '제품설명', max_length = 500)
    discount_rate: float = Field(default = 0.0, description = '제품 할인율', ge = 0, le = 1)
    created_at: datetime = Field(default_factory = datetime.now, description = '제품식별코드')

In [17]:
# 형태가 달라도 자동 형변환 가능시 자동 파싱 / 필수값이 아니면 생략 가능(기본값 자동 설정)
prod = Product(
    code = 'pr102',
    name = '물',
    price = 300_000,
    quantity = 50,
    description = '삼다수',
    discount_rate = 0.95
)

print(prod)

code='pr102' name='물' price=300000.0 quantity=50 description='삼다수' discount_rate=0.95 created_at=datetime.datetime(2026, 9, 2, 16, 22, 29, 409512)


In [18]:
# json 변환
prod_json_str = prod.model_dump_json()
print(prod_json_str)

{"code":"pr102","name":"물","price":300000.0,"quantity":50,"description":"삼다수","discount_rate":0.95,"created_at":"2026-09-02T16:22:29.409512"}


In [19]:
# Pydantic 객체 -> dict로 변환
prod_dict = prod.model_dump()
prod_dict

{'code': 'pr102',
 'name': '물',
 'price': 300000.0,
 'quantity': 50,
 'description': '삼다수',
 'discount_rate': 0.95,
 'created_at': datetime.datetime(2026, 9, 2, 16, 22, 29, 409512)}

In [23]:
# JSON -> Pydantic 객체로 변환
json_str = '''
{
    "code": "pr102",
    "name": "물",
    "price": 300000.0,
    "quantity": 50,
    "description": "삼다수",
    "discount_rate": 0.95,
    "created_at": "2026-09-02T16:22:29.409512"
}
'''

prod_from_json = Product.model_validate_json(json_str)

print(prod_from_json, type(prod_from_json))

code='pr102' name='물' price=300000.0 quantity=50 description='삼다수' discount_rate=0.95 created_at=datetime.datetime(2026, 9, 2, 16, 22, 29, 409512) <class '__main__.Product'>


In [24]:
from datetime import datetime


data = {
    'code': 'pr102',
    'name': '물',
    'price': 300000.0,
    'quantity': 50,
    'description': '삼다수',
    'discount_rate': 0.95,
    'created_at': datetime(
        2026, 9, 2, 16, 22, 29, 409512
    )
}

prod_from_dict = Product(**data)

print(prod_from_dict)

code='pr102' name='물' price=300000.0 quantity=50 description='삼다수' discount_rate=0.95 created_at=datetime.datetime(2026, 9, 2, 16, 22, 29, 409512)
